# Ingest Fundamentals + Dividends from Merolagani

> Respectful scraping workflow: request pacing, local HTML cache, and idempotent DB writes.

This notebook populates both `fundamentals` and `dividends` for broad equity coverage by default.
You can optionally set sector filters in the config cell for faster, targeted refresh runs.

In [2]:
from __future__ import annotations

import re
import sys
import time
import sqlite3
from pathlib import Path
from typing import Any
from urllib.parse import urlencode

import requests
from bs4 import BeautifulSoup

# Ensure local package imports work when running from notebooks/.
sys.path.append(str(Path.cwd().parent))

from nepse_analyst.config import BASE_DIR, DB_PATH, SCRAPE_DELAY_SEC

PROJECT_ROOT = Path(BASE_DIR)
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "news_cache"
RAW_DIR.mkdir(parents=True, exist_ok=True)

MEROLAGANI_BASE_URL = "https://merolagani.com/CompanyDetail.aspx"
REQUEST_TIMEOUT_SEC = 20
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# Leave as None to ingest all eligible sectors.
INCLUDE_ONLY_SECTORS: tuple[str, ...] | None = None

# Exclude instrument classes that do not have comparable company fundamentals.
EXCLUDED_SECTORS = (
    "Govt. Bonds",
    "Mutual Fund",
    "Corp. Debentures",
)

FUNDAMENTAL_COLUMNS = [
    "symbol",
    "fiscal_year",
    "eps",
    "pe_ratio",
    "book_value",
    "roe",
    "net_profit",
    "total_assets",
    "revenue",
]

DIVIDEND_COLUMNS = [
    "symbol",
    "fiscal_year",
    "cash_dividend",
    "bonus_shares",
    "book_close_date",
    "announced_date",
]

print(f"DB path: {DB_PATH}")
print(f"Raw cache dir: {RAW_DIR}")
print(f"Configured scrape delay: {SCRAPE_DELAY_SEC}s")

DB path: /home/meutsabdahal/Desktop/nepse-analyst/data/processed/nepse.db
Raw cache dir: /home/meutsabdahal/Desktop/nepse-analyst/data/raw/news_cache
Configured scrape delay: 3s


In [3]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

base_sql = """
SELECT symbol, name, sector
FROM companies
"""

if INCLUDE_ONLY_SECTORS:
    placeholders = ",".join(["?"] * len(INCLUDE_ONLY_SECTORS))
    companies = cur.execute(
        base_sql + f" WHERE sector IN ({placeholders}) ORDER BY symbol",
        INCLUDE_ONLY_SECTORS,
    ).fetchall()
    print(
        f"Targeted companies found: {len(companies)} "
        f"(sectors={', '.join(INCLUDE_ONLY_SECTORS)})"
    )
else:
    placeholders = ",".join(["?"] * len(EXCLUDED_SECTORS))
    companies = cur.execute(
        base_sql + f" WHERE sector NOT IN ({placeholders}) ORDER BY symbol",
        EXCLUDED_SECTORS,
    ).fetchall()
    print(
        f"Eligible companies found: {len(companies)} "
        f"(excluding sectors={', '.join(EXCLUDED_SECTORS)})"
    )

[dict(row) for row in companies[:10]]

Eligible companies found: 421 (excluding sectors=Govt. Bonds, Mutual Fund, Corp. Debentures)


[{'symbol': 'ACLBSL',
  'name': 'Aarambha Chautari Laghubitta Bittiya Sanstha Limited',
  'sector': 'Microfinance'},
 {'symbol': 'ACLBSLP',
  'name': 'Aarambha Chautari Laghubitta Bittiya Sanstha Limited Promoter Share',
  'sector': 'Others'},
 {'symbol': 'ADBL',
  'name': 'Agriculture Development Bank Limited',
  'sector': 'Commercial Banks'},
 {'symbol': 'ADLB',
  'name': 'Adarsha Laghubitta Bittiya Sanstha Limited',
  'sector': 'Microfinance'},
 {'symbol': 'AHL', 'name': 'Asian Hydropower Limited', 'sector': 'Hydropower'},
 {'symbol': 'AHPC',
  'name': 'Arun Valley Hydropower Development Co. Ltd.',
  'sector': 'Hydropower'},
 {'symbol': 'AKJCL',
  'name': 'Ankhu Khola Jalvidhyut Company Ltd',
  'sector': 'Hydropower'},
 {'symbol': 'AKPL', 'name': 'Arun Kabeli Power Ltd.', 'sector': 'Hydropower'},
 {'symbol': 'ALBSL',
  'name': 'Asha Laghubitta Bittiya Sanstha Limited',
  'sector': 'Microfinance'},
 {'symbol': 'ALBSLP',
  'name': 'Asha Laghubitta Bittiya Sanstha Ltd Promoter Share',


In [4]:
FUNDAMENTAL_KEYWORDS = {
    "eps": ["eps", "earning per share", "earnings per share"],
    "pe_ratio": ["p/e", "p e ratio", "pe ratio", "price earnings"],
    "book_value": ["book value", "book value per share", "bvps"],
    "roe": ["roe", "return on equity"],
    "net_profit": ["net profit", "profit after tax", "pat"],
    "total_assets": ["total assets", "asset"],
    "revenue": ["revenue", "turnover", "income"],
}

DIVIDEND_TABLE_HINTS = ["dividend", "bonus", "book close", "announced", "distribution"]

def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def slug(text: str) -> str:
    return re.sub(r"[^a-z0-9%]+", " ", (text or "").lower()).strip()


def to_float(text: str | None) -> float | None:
    if text is None:
        return None
    raw = normalize_ws(text).replace(",", "")
    if raw.lower() in {"", "-", "--", "n/a", "na", "none"}:
        return None
    m = re.search(r"-?\d+(?:\.\d+)?", raw)
    if not m:
        return None
    try:
        return float(m.group(0))
    except ValueError:
        return None


def _normalize_year_token(tok: str) -> int | None:
    tok = tok.strip()
    if not tok.isdigit():
        return None
    n = int(tok)
    if len(tok) == 4:
        return n
    if len(tok) == 3:
        # Merolagani often uses FY shorthand like 082-083 for 2082/83.
        return 2000 + n
    if len(tok) == 2 and n >= 70:
        # Conservative handling for Nepali FY shorthand; avoid Gregorian MM false-positives.
        return 2000 + n
    return None


def normalize_fiscal_year(text: str | None) -> str | None:
    if not text:
        return None
    cleaned = normalize_ws(text)

    m = re.search(r"(?:FY\s*[:\-]?\s*)?(\d{2,4})\s*[/\-]\s*(\d{2,4})", cleaned, flags=re.I)
    if m:
        left = _normalize_year_token(m.group(1))
        right = _normalize_year_token(m.group(2))
        if left is not None and right is not None:
            return f"{left:04d}/{right % 100:02d}"

    m2 = re.search(r"FY\s*[:\-]?\s*(\d{2,4})", cleaned, flags=re.I)
    if m2:
        left = _normalize_year_token(m2.group(1))
        if left is not None:
            return f"{left:04d}/{(left + 1) % 100:02d}"

    return None


def extract_fiscal_year_from_text(text: str | None) -> str | None:
    if not text:
        return None
    # Prefer FY-tagged tokens to avoid parsing trade-date fragments like 2026/03.
    fy = normalize_fiscal_year(text)
    if fy:
        return fy
    if "fy" in (text or "").lower():
        m = re.search(r"(\d{2,4}\s*[/\-]\s*\d{2,4})", text)
        if m:
            return normalize_fiscal_year(f"FY:{m.group(1)}")
    return None


def build_company_url(symbol: str) -> str:
    params = urlencode({"symbol": symbol.upper()})
    return f"{MEROLAGANI_BASE_URL}?{params}"


def load_soup_from_html(html: str) -> BeautifulSoup:
    return BeautifulSoup(html, "html.parser")


def fetch_company_html(
    session: requests.Session,
    symbol: str,
    last_request_ts: float | None,
    force_refresh: bool = False,
) -> tuple[str, float | None, bool]:
    cache_path = RAW_DIR / f"merolagani_{symbol.upper()}.html"
    if cache_path.exists() and not force_refresh:
        return cache_path.read_text(encoding="utf-8", errors="ignore"), last_request_ts, True

    if last_request_ts is not None:
        elapsed = time.time() - last_request_ts
        if elapsed < SCRAPE_DELAY_SEC:
            time.sleep(SCRAPE_DELAY_SEC - elapsed)

    resp = session.get(build_company_url(symbol), headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC)
    resp.raise_for_status()
    html = resp.text
    cache_path.write_text(html, encoding="utf-8")
    return html, time.time(), False


def metric_key(row_label: str) -> str | None:
    s = slug(row_label)
    for key, variants in FUNDAMENTAL_KEYWORDS.items():
        if any(v in s for v in variants):
            return key
    return None


def _new_fundamentals_record(symbol: str, fiscal_year: str) -> dict[str, Any]:
    return {
        "symbol": symbol,
        "fiscal_year": fiscal_year,
        "eps": None,
        "pe_ratio": None,
        "book_value": None,
        "roe": None,
        "net_profit": None,
        "total_assets": None,
        "revenue": None,
    }


def _new_dividend_record(symbol: str, fiscal_year: str) -> dict[str, Any]:
    return {
        "symbol": symbol,
        "fiscal_year": fiscal_year,
        "cash_dividend": None,
        "bonus_shares": None,
        "book_close_date": None,
        "announced_date": None,
    }


def extract_table_matrix(table_tag: Any) -> tuple[list[str], list[list[str]]]:
    rows = table_tag.find_all("tr")
    matrix: list[list[str]] = []
    for r in rows:
        cells = r.find_all(["th", "td"])
        vals = [normalize_ws(c.get_text(" ", strip=True)) for c in cells]
        if any(vals):
            matrix.append(vals)
    if not matrix:
        return [], []
    return matrix[0], matrix[1:] if len(matrix) > 1 else []


def parse_fundamentals(soup: BeautifulSoup, symbol: str) -> list[dict[str, Any]]:
    records_by_fy: dict[str, dict[str, Any]] = {}
    default_fy: str | None = None

    # 1) Parse key/value rows from the main accordion summary table.
    acc = soup.find("table", id="accordion")
    if acc:
        for tr in acc.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) < 2:
                continue
            label = normalize_ws(cells[0].get_text(" ", strip=True))
            value_text = normalize_ws(cells[1].get_text(" ", strip=True))
            mk = metric_key(label)
            if not mk:
                continue

            row_fy = extract_fiscal_year_from_text(value_text)
            if mk == "eps" and row_fy:
                default_fy = row_fy
            fy = row_fy or default_fy
            if not fy:
                continue
            rec = records_by_fy.setdefault(fy, _new_fundamentals_record(symbol, fy))
            rec[mk] = to_float(value_text)

    # 2) Fallback parser for wide FY-column tables when available.
    for table in soup.find_all("table"):
        header, body = extract_table_matrix(table)
        if not header or len(header) < 2:
            continue

        fiscal_cols: list[tuple[int, str]] = []
        for idx, col in enumerate(header[1:], start=1):
            fy = extract_fiscal_year_from_text(col)
            if fy:
                fiscal_cols.append((idx, fy))

        if not fiscal_cols:
            continue

        for _, fy in fiscal_cols:
            records_by_fy.setdefault(fy, _new_fundamentals_record(symbol, fy))

        for row in body:
            if len(row) < 2:
                continue
            mk = metric_key(row[0])
            if not mk:
                continue
            for col_idx, fy in fiscal_cols:
                if col_idx < len(row):
                    val = to_float(row[col_idx])
                    if val is not None:
                        records_by_fy[fy][mk] = val

    return sorted(records_by_fy.values(), key=lambda x: x["fiscal_year"], reverse=True)


def parse_dividends(soup: BeautifulSoup, symbol: str) -> list[dict[str, Any]]:
    out: dict[str, dict[str, Any]] = {}

    def upsert_value(fy: str, field: str, val: Any) -> None:
        rec = out.setdefault(fy, _new_dividend_record(symbol, fy))
        if val is not None:
            rec[field] = val

    # 1) Parse summary rows in accordion: % Dividend, % Bonus with inline FY.
    acc = soup.find("table", id="accordion")
    if acc:
        for tr in acc.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) < 2:
                continue
            label = slug(cells[0].get_text(" ", strip=True))
            value_text = normalize_ws(cells[1].get_text(" ", strip=True))
            fy = extract_fiscal_year_from_text(value_text)
            if not fy:
                continue
            if "dividend" in label:
                upsert_value(fy, "cash_dividend", to_float(value_text))
            elif "bonus" in label:
                upsert_value(fy, "bonus_shares", to_float(value_text))

        # 2) Parse collapsed panel rows (dividend-panel / bonus-panel) when populated.
        for panel_id, target_field in [("dividend-panel", "cash_dividend"), ("bonus-panel", "bonus_shares")]:
            panel = acc.find("tr", id=panel_id)
            if not panel:
                continue
            for r in panel.find_all("tr"):
                vals = [normalize_ws(c.get_text(" ", strip=True)) for c in r.find_all(["th", "td"])]
                if not vals:
                    continue
                if any(v.lower() in {"#", "value", "fiscal year"} for v in vals):
                    continue
                fy = None
                num = None
                for v in vals:
                    fy = fy or extract_fiscal_year_from_text(v)
                    if num is None and ("%" in v or re.search(r"\d", v)):
                        num = to_float(v)
                if fy and num is not None:
                    upsert_value(fy, target_field, num)

    # 3) Generic fallback for dividend-like tables if they include FY/value columns.
    for table in soup.find_all("table"):
        table_text = slug(table.get_text(" ", strip=True))
        if not any(hint in table_text for hint in DIVIDEND_TABLE_HINTS):
            continue

        header, body = extract_table_matrix(table)
        if not header or not body:
            continue
        hdr = [slug(h) for h in header]

        def col_index(*needles: str) -> int | None:
            for i, name in enumerate(hdr):
                if any(n in name for n in needles):
                    return i
            return None

        fy_idx = col_index("fiscal year", "fy")
        val_idx = col_index("value", "cash", "dividend", "bonus", "stock")
        if fy_idx is None or val_idx is None:
            continue

        target_field = "bonus_shares" if "bonus" in table_text else "cash_dividend"
        for row in body:
            if fy_idx >= len(row) or val_idx >= len(row):
                continue
            fy = extract_fiscal_year_from_text(row[fy_idx])
            if fy:
                upsert_value(fy, target_field, to_float(row[val_idx]))

    return sorted(out.values(), key=lambda x: x["fiscal_year"], reverse=True)

In [5]:
session = requests.Session()
last_request_ts: float | None = None

fundamental_records: list[dict[str, Any]] = []
dividend_records: list[dict[str, Any]] = []
errors: list[dict[str, str]] = []
request_count = 0
cache_hits = 0

for row in companies:
    symbol = row["symbol"]

    try:
        html, last_request_ts, from_cache = fetch_company_html(
            session=session,
            symbol=symbol,
            last_request_ts=last_request_ts,
            force_refresh=False,
        )
        cache_hits += int(from_cache)
        request_count += int(not from_cache)

        soup = load_soup_from_html(html)

        f_rows = parse_fundamentals(soup, symbol)
        d_rows = parse_dividends(soup, symbol)

        fundamental_records.extend(f_rows)
        dividend_records.extend(d_rows)

        print(
            f"{symbol}: fundamentals={len(f_rows)} | dividends={len(d_rows)} | "
            f"source={'cache' if from_cache else 'web'}"
        )
    except Exception as exc:  # noqa: BLE001
        errors.append({"symbol": symbol, "error": str(exc)})
        print(f"[ERROR] {symbol}: {exc}")

print("\n--- Scrape summary ---")
print(f"Companies processed: {len(companies)}")
print(f"Network requests made: {request_count}")
print(f"Cache hits: {cache_hits}")
print(f"Fundamental rows extracted: {len(fundamental_records)}")
print(f"Dividend rows extracted: {len(dividend_records)}")
print(f"Errors: {len(errors)}")
errors[:5]

cur.executescript(
    """
    CREATE UNIQUE INDEX IF NOT EXISTS ux_fundamentals_symbol_fy
    ON fundamentals(symbol, fiscal_year);

    CREATE UNIQUE INDEX IF NOT EXISTS ux_dividends_symbol_fy
    ON dividends(symbol, fiscal_year);
    """
)

fundamental_upsert_sql = """
INSERT INTO fundamentals (
    symbol, fiscal_year, eps, pe_ratio, book_value, roe, net_profit, total_assets, revenue
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(symbol, fiscal_year) DO UPDATE SET
    eps = excluded.eps,
    pe_ratio = excluded.pe_ratio,
    book_value = excluded.book_value,
    roe = excluded.roe,
    net_profit = excluded.net_profit,
    total_assets = excluded.total_assets,
    revenue = excluded.revenue;
"""

dividend_upsert_sql = """
INSERT INTO dividends (
    symbol, fiscal_year, cash_dividend, bonus_shares, book_close_date, announced_date
) VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(symbol, fiscal_year) DO UPDATE SET
    cash_dividend = excluded.cash_dividend,
    bonus_shares = excluded.bonus_shares,
    book_close_date = excluded.book_close_date,
    announced_date = excluded.announced_date;
"""

fundamental_payload = [
    tuple(rec.get(c) for c in FUNDAMENTAL_COLUMNS)
    for rec in fundamental_records
    if rec.get("fiscal_year")
]

dividend_payload = [
    tuple(rec.get(c) for c in DIVIDEND_COLUMNS)
    for rec in dividend_records
    if rec.get("fiscal_year")
]

symbols = [r["symbol"] for r in companies]
if symbols:
    placeholders = ",".join(["?"] * len(symbols))
    cur.execute(f"DELETE FROM fundamentals WHERE symbol IN ({placeholders})", symbols)
    cur.execute(f"DELETE FROM dividends WHERE symbol IN ({placeholders})", symbols)

if fundamental_payload:
    cur.executemany(fundamental_upsert_sql, fundamental_payload)
if dividend_payload:
    cur.executemany(dividend_upsert_sql, dividend_payload)

conn.commit()

print(f"Upserted fundamentals rows: {len(fundamental_payload)}")
print(f"Upserted dividends rows: {len(dividend_payload)}")

ACLBSL: fundamentals=1 | dividends=0 | source=cache
ACLBSLP: fundamentals=0 | dividends=0 | source=cache
ADBL: fundamentals=1 | dividends=10 | source=cache
ADLB: fundamentals=1 | dividends=1 | source=cache
AHL: fundamentals=1 | dividends=0 | source=cache
AHPC: fundamentals=1 | dividends=13 | source=cache
AKJCL: fundamentals=1 | dividends=0 | source=cache
AKPL: fundamentals=1 | dividends=3 | source=cache
ALBSL: fundamentals=1 | dividends=6 | source=cache
ALBSLP: fundamentals=0 | dividends=0 | source=cache
ALICL: fundamentals=1 | dividends=10 | source=cache
ALICLP: fundamentals=0 | dividends=3 | source=cache
ANLB: fundamentals=1 | dividends=3 | source=cache
ANLBP: fundamentals=0 | dividends=0 | source=cache
API: fundamentals=1 | dividends=5 | source=cache
AVYAN: fundamentals=1 | dividends=0 | source=cache
AVYANP: fundamentals=0 | dividends=0 | source=cache
BARUN: fundamentals=1 | dividends=3 | source=cache
BBC: fundamentals=1 | dividends=1 | source=cache
BEDC: fundamentals=1 | dividends=

## Data quality notes to expect

> These are expected and handled by code below:
- Missing EPS/P-E/ROE in older years is stored as NULL (not dropped).
- Fiscal year strings can vary (e.g., `FY 2079/80`, `2079-80`, `2079/2080`) and are normalized to `YYYY/YY` where possible.
- Re-runs are deduplicated with unique indexes and upsert logic (`ON CONFLICT(symbol, fiscal_year)`).

In [6]:
sample_fundamentals = cur.execute(
    """
    SELECT symbol, fiscal_year, eps, pe_ratio, book_value, roe, net_profit, total_assets, revenue
    FROM fundamentals
    ORDER BY symbol, fiscal_year DESC
    LIMIT 10
    """
).fetchall()

sample_dividends = cur.execute(
    """
    SELECT symbol, fiscal_year, cash_dividend, bonus_shares, book_close_date, announced_date
    FROM dividends
    ORDER BY symbol, fiscal_year DESC
    LIMIT 10
    """
).fetchall()

print("Sample fundamentals:")
display([dict(r) for r in sample_fundamentals])

print("Sample dividends:")
display([dict(r) for r in sample_dividends])

conn.close()
print("DB connection closed.")

Sample fundamentals:


[{'symbol': 'ACLBSL',
  'fiscal_year': '2082/83',
  'eps': 32.57,
  'pe_ratio': 32.08,
  'book_value': 169.98,
  'roe': None,
  'net_profit': None,
  'total_assets': None,
  'revenue': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2082/83',
  'eps': 5.33,
  'pe_ratio': 62.1,
  'book_value': 224.37,
  'roe': None,
  'net_profit': None,
  'total_assets': None,
  'revenue': None},
 {'symbol': 'ADLB',
  'fiscal_year': '2079/80',
  'eps': 1.34,
  'pe_ratio': 880.6,
  'book_value': 126.13,
  'roe': None,
  'net_profit': None,
  'total_assets': None,
  'revenue': None},
 {'symbol': 'AHL',
  'fiscal_year': '2082/83',
  'eps': -17.0,
  'pe_ratio': -30.55,
  'book_value': 77.34,
  'roe': None,
  'net_profit': None,
  'total_assets': None,
  'revenue': None},
 {'symbol': 'AHPC',
  'fiscal_year': '2082/83',
  'eps': 4.95,
  'pe_ratio': 64.65,
  'book_value': 114.9,
  'roe': None,
  'net_profit': None,
  'total_assets': None,
  'revenue': None},
 {'symbol': 'AKJCL',
  'fiscal_year': '2082/83',
  'ep

Sample dividends:


[{'symbol': 'ADBL',
  'fiscal_year': '2081/82',
  'cash_dividend': 1.0,
  'bonus_shares': 3.25,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2080/81',
  'cash_dividend': 2.0,
  'bonus_shares': None,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2078/79',
  'cash_dividend': 4.0,
  'bonus_shares': None,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2077/78',
  'cash_dividend': 6.0,
  'bonus_shares': None,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2076/77',
  'cash_dividend': 7.0,
  'bonus_shares': None,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2075/76',
  'cash_dividend': 8.0,
  'bonus_shares': None,
  'book_close_date': None,
  'announced_date': None},
 {'symbol': 'ADBL',
  'fiscal_year': '2073/74',
  'cash_dividend': 10.0,
  'bonus_shares': None,
  '

DB connection closed.


In [7]:
from pathlib import Path
import sqlite3

cache_files = sorted(Path(RAW_DIR).glob("merolagani_*.html"))

print("Run summary:")
print(f"Companies processed: {len(companies)}")
print(f"Network requests made: {request_count}")
print(f"Cache hits: {cache_hits}")
print(f"Extracted fundamentals records (pre-upsert): {len(fundamental_records)}")
print(f"Extracted dividends records (pre-upsert): {len(dividend_records)}")
print(f"Errors captured: {len(errors)}")
print(f"Cached html files present: {len(cache_files)}")
print("First 5 errors:")
display(errors[:5])

conn_check = sqlite3.connect(DB_PATH)
conn_check.row_factory = sqlite3.Row
cur_check = conn_check.cursor()

fund_count = cur_check.execute("SELECT COUNT(*) AS c FROM fundamentals").fetchone()["c"]
div_count = cur_check.execute("SELECT COUNT(*) AS c FROM dividends").fetchone()["c"]
nonnull_eps = cur_check.execute("SELECT COUNT(*) AS c FROM fundamentals WHERE eps IS NOT NULL").fetchone()["c"]
distinct_fund_symbols = cur_check.execute("SELECT COUNT(DISTINCT symbol) AS c FROM fundamentals").fetchone()["c"]
distinct_div_symbols = cur_check.execute("SELECT COUNT(DISTINCT symbol) AS c FROM dividends").fetchone()["c"]

print("\nDatabase summary:")
print(f"fundamentals rows: {fund_count}")
print(f"dividends rows: {div_count}")
print(f"fundamentals rows with non-null EPS: {nonnull_eps}")
print(f"distinct symbols in fundamentals: {distinct_fund_symbols}")
print(f"distinct symbols in dividends: {distinct_div_symbols}")

conn_check.close()

Run summary:
Companies processed: 421
Network requests made: 0
Cache hits: 421
Extracted fundamentals records (pre-upsert): 292
Extracted dividends records (pre-upsert): 1461
Errors captured: 0
Cached html files present: 421
First 5 errors:


[]


Database summary:
fundamentals rows: 292
dividends rows: 1461
fundamentals rows with non-null EPS: 292
distinct symbols in fundamentals: 292
distinct symbols in dividends: 267
